# 01 — AI 1 Data EDA & Quality Audit

**Mục tiêu:** kiểm tra dữ liệu *v2-original-ids* sau chuẩn hóa: số dòng, schema, ID/FK, tọa độ, cuisine, dietary, giá synthetic và coverage. Dùng **Parquet thực tế** của project; notebook không thay đổi dữ liệu gốc. Các con số chỉ xuất hiện sau khi chạy notebook trên máy m.

**Giới hạn:** cuisine được gán nhãn không đồng nghĩa đúng thực tế; giá menu là ước tính synthetic, không phải giá quán xác nhận.

In [ ]:
import sys
import pyarrow
import pandas as pd

print("Python:", sys.executable)
print("PyArrow:", pyarrow.__version__)

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

ROOT = Path.cwd()

PROCESSED = ROOT / "data" / "processed"
REPORTS = ROOT / "reports" / "ai1"
REPORTS.mkdir(parents=True, exist_ok=True)

restaurants = pd.read_parquet(PROCESSED / "restaurants.parquet")
dishes = pd.read_parquet(PROCESSED / "dishes.parquet")
menus = pd.read_parquet(PROCESSED / "menu_items.parquet")

print("ROOT:", ROOT)
print("Rows:", {
    "restaurants": len(restaurants),
    "dishes": len(dishes),
    "menu_items": len(menus)
})

## 1. Schema, null và ID

In [ ]:
for name, df in [("restaurants",restaurants),("dishes",dishes),("menu_items",menus)]:
    print(f"\n{name}: {df.shape}")
    display(pd.DataFrame({'dtype':df.dtypes.astype(str), 'missing':df.isna().sum(), 'missing_pct':(df.isna().mean()*100).round(2)}))
keys={'restaurants':(restaurants,'food_place_id'),'dishes':(dishes,'dish_id'),'menu_items':(menus,'menu_item_id')}
checks={}
for name,(df,key) in keys.items():
    checks[f'{name}_null_ids']=int(df[key].isna().sum())
    checks[f'{name}_duplicate_ids']=int(df[key].duplicated().sum())
checks['orphan_menu_restaurants']=int((~menus.food_place_id.isin(restaurants.food_place_id)).sum())
checks['orphan_menu_dishes']=int((~menus.dish_id.isin(dishes.dish_id)).sum())
display(pd.Series(checks,name='count').to_frame())
assert all(v==0 for v in checks.values()), 'ID/FK checks failed' 

## 2. Geo và khả năng lọc

In [ ]:
lat=pd.to_numeric(restaurants.latitude,errors='coerce');lon=pd.to_numeric(restaurants.longitude,errors='coerce')
geo={'missing_coordinates':int((lat.isna()|lon.isna()).sum()),'invalid_lat':int((lat.notna()&~lat.between(-90,90)).sum()),'invalid_lon':int((lon.notna()&~lon.between(-180,180)).sum())}
if 'is_recommendable' in restaurants:
    geo['recommendable_true']=int(restaurants.is_recommendable.astype('string').str.lower().eq('true').sum())
display(pd.Series(geo,name='count').to_frame())

## 3. Cuisine coverage — tách nhãn gốc và supported

In [ ]:
def known(s):
    return s.astype('string').str.strip().str.lower().notna() & ~s.astype('string').str.strip().str.lower().isin(['unknown','other','nan','none',''])
coverage={}
for name,df,cols in [('restaurants',restaurants,['normalized_cuisine','supported_cuisine']),('dishes',dishes,['cuisine_origin','supported_cuisine'])]:
    for col in cols:
        if col in df: coverage[f'{name}_{col}_specific']=int(known(df[col]).sum())
        else: coverage[f'{name}_{col}_specific']='MISSING COLUMN'
display(pd.Series(coverage,name='count').to_frame())
for name,df in [('restaurants',restaurants),('dishes',dishes)]:
    if 'supported_cuisine' in df:
        print(name);display(df.supported_cuisine.fillna('unknown').value_counts().head(15).to_frame('count'))
if 'cuisine_confidence' in dishes:
    display(dishes.cuisine_confidence.fillna('unknown').value_counts().to_frame('dish_count'))

## 4. Dietary, menu price và độ bao phủ

In [ ]:
if 'is_vegetarian' in dishes:
    display(dishes.is_vegetarian.astype('string').str.lower().fillna('unknown').value_counts(dropna=False).to_frame('dish_count'))
price=pd.to_numeric(menus.price_vnd if 'price_vnd' in menus else menus.estimated_price_vnd,errors='coerce')
price_report={'null_price':int(price.isna().sum()),'nonpositive_price':int((price<=0).sum()),'min_vnd':float(price.min()),'median_vnd':float(price.median()),'p95_vnd':float(price.quantile(.95)),'max_vnd':float(price.max())}
if 'is_synthetic' in menus: price_report['synthetic_rows']=int(menus.is_synthetic.astype('string').str.lower().eq('true').sum())
if 'price_is_verified' in menus: price_report['verified_rows']=int(menus.price_is_verified.astype('string').str.lower().eq('true').sum())
display(pd.Series(price_report,name='value').to_frame())
print('Menu items / restaurant:',menus.groupby('food_place_id').size().describe().to_dict())

## 5. Xuất báo cáo và tiêu chí hoàn thành

In [ ]:
audit={'dataset_version':'v2-original-ids','rows':{k:len(v[0]) for k,v in keys.items()},'integrity':checks,'geo':geo,'cuisine':coverage,'prices':price_report,'limitations':['Menu prices synthetic; not verified','Cuisine coverage and confidence vary','Coordinates imply straight-line distance only']}
out=REPORTS/'eda_quality_report.json';out.write_text(json.dumps(audit,ensure_ascii=False,indent=2,default=str),encoding='utf-8')
print('Saved:',out)
print('PASS integrity:',all(v==0 for v in checks.values()))
print('NOTE: quality of cuisine labels requires manual review; this notebook measures coverage, not accuracy.')